# Bandwidth, Delay and the Window You Need

Companion notebook for the [Latency, Bandwidth and the Delay Product](https://codemia.ai/courses/networking_for_backend_engineers/net_latency_bandwidth_and_the_delay_product) lesson on Codemia.

**What you will build**: a calculator and a set of plots that show why a connection with plenty of
bandwidth still transfers slowly, and exactly how large a window has to be before the bandwidth
becomes reachable.

The one equation behind all of it is the bandwidth-delay product: a path with a given rate and a
given round trip holds a fixed number of bytes in flight. Send less than that and the sender spends
most of its time waiting for acknowledgements. The window is not a tuning knob for throughput, it
*is* the throughput, once you divide it by the round trip.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

KB = 1024
MB = 1024 * 1024

def bdp_bytes(mbps, rtt_ms):
    """Bytes in flight on a full path: rate x round trip, converted to bytes."""
    bits_per_second = mbps * 1_000_000          # link rates are decimal, not binary
    seconds = rtt_ms / 1000.0
    return bits_per_second * seconds / 8.0

print(f'1 Gbit/s at 1ms   -> {bdp_bytes(1000, 1):>12,.0f} bytes')
print(f'1 Gbit/s at 80ms  -> {bdp_bytes(1000, 80):>12,.0f} bytes')
print(f'100 Mbit/s at 200ms -> {bdp_bytes(100, 200):>12,.0f} bytes')

## Concept recap

Two numbers describe a path and they are independent. Bandwidth is how many bits per second the
narrowest link can carry. Latency is how long one bit takes to get there and back. Neither improves
the other: a satellite link can have enormous bandwidth and a round trip of 600ms, and a short
copper run can have 10 Mbit/s and a round trip of 1ms.

A sender with a window of `W` bytes can have `W` bytes unacknowledged. After that it waits a round
trip. So the ceiling on throughput is `W / RTT`, whatever the link underneath can do. The window
that removes the ceiling is the bandwidth-delay product.

## The window a path needs

In [ ]:
paths = [
    ('same rack',        10_000,   0.2),
    ('same datacentre',  10_000,   0.5),
    ('same region',       1_000,     2),
    ('cross-country',     1_000,    35),
    ('transatlantic',     1_000,    80),
    ('transpacific',      1_000,   140),
    ('satellite',           100,   600),
]

print(f"{'path':<18}{'Mbit/s':>8}{'RTT ms':>8}{'window needed':>16}")
for name, mbps, rtt in paths:
    need = bdp_bytes(mbps, rtt)
    print(f'{name:<18}{mbps:>8}{rtt:>8}{need/KB:>13,.0f} KB')

Notice the transatlantic row. A gigabit path with an 80ms round trip needs ten megabytes in flight
to stay busy. No operating system ships with a default that large, which is why a single connection
across an ocean never reaches the line rate however fast both ends are.

## What a fixed window actually gets you

In [ ]:
def throughput_mbps(window_bytes, rtt_ms, link_mbps):
    """A window caps throughput at window/RTT; the link caps it too. Take the lower."""
    window_limit = window_bytes * 8 / (rtt_ms / 1000.0) / 1_000_000
    return min(window_limit, link_mbps)

windows = [(64 * KB, '64 KB (no window scaling)'),
           (256 * KB, '256 KB'),
           (1 * MB, '1 MB'),
           (16 * MB, '16 MB')]

rtts = np.array([0.5, 2, 10, 35, 80, 140, 300])

print(f"{'RTT ms':>8}" + ''.join(f'{label:>28}' for _, label in windows))
for rtt in rtts:
    row = ''.join(f'{throughput_mbps(w, rtt, 1000):>25,.1f}   ' for w, _ in windows)
    print(f'{rtt:>8}' + row)
print()
print('(figures are Mbit/s on a 1 Gbit/s path)')

The first column is the historical default before window scaling was negotiated, and it is worth
looking at: 64 KB over an 80ms round trip is about 6.5 Mbit/s. A gigabit path, a receiver that can
take it, and 6.5 Mbit/s of throughput. Nothing is broken and nothing is congested.

In [ ]:
fine_rtts = np.linspace(0.5, 300, 400)

plt.figure(figsize=(9, 5))
for window, label in windows:
    series = [throughput_mbps(window, r, 1000) for r in fine_rtts]
    plt.plot(fine_rtts, series, label=label)

plt.axhline(1000, color='grey', linestyle=':', linewidth=1)
plt.text(5, 1020, '1 Gbit/s link rate', color='grey', fontsize=9)
plt.xlabel('round trip time (ms)')
plt.ylabel('throughput ceiling (Mbit/s)')
plt.title('A window is a throughput ceiling, divided by the round trip')
plt.yscale('log')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Every curve is the same shape because every curve is the same equation. The flat segment on the
left is where the link is the limit; the sloped segment is where the window is. Moving the knee to
the right is the entire purpose of window scaling.

## How long a transfer takes, including the start

In [ ]:
def transfer_seconds(size_bytes, mbps, rtt_ms, initial_window_segments=10, mss=1460):
    """Slow start doubles the window each round trip until the path is full."""
    bdp = bdp_bytes(mbps, rtt_ms)
    window = initial_window_segments * mss
    sent = 0
    rounds = 0
    while sent < size_bytes and window < bdp:
        sent += window
        window *= 2
        rounds += 1
    remaining = max(0, size_bytes - sent)
    steady_seconds = remaining * 8 / (mbps * 1_000_000)
    return rounds * rtt_ms / 1000.0 + steady_seconds + rtt_ms / 1000.0

for size, label in [(14 * KB, 'small page'), (200 * KB, 'bundle'), (5 * MB, 'image'), (100 * MB, 'archive')]:
    local = transfer_seconds(size, 1000, 2)
    far = transfer_seconds(size, 1000, 140)
    print(f'{label:<12}{size/KB:>9,.0f} KB   local {local*1000:>8,.1f} ms   transpacific {far*1000:>9,.1f} ms   ratio {far/local:>5.1f}x')

The ratio column is the point. A small response is almost entirely round trips, so distance
multiplies it directly. A large transfer spends most of its time in steady state, where the link
rate dominates and distance matters much less. This is why moving a static asset to an edge helps
a page far more than moving a bulk download does.

## Variations: what to change when a transfer is slow

In [ ]:
def report(size_bytes, mbps, rtt_ms, window_bytes):
    ceiling = throughput_mbps(window_bytes, rtt_ms, mbps)
    seconds = size_bytes * 8 / (ceiling * 1_000_000)
    return ceiling, seconds

size = 50 * MB
base = report(size, 1000, 80, 256 * KB)
print(f'baseline            256 KB window, 80ms, 1 Gbit/s -> {base[0]:>7,.1f} Mbit/s, {base[1]:>6,.1f} s')

for label, args in [
    ('double the bandwidth', (size, 2000, 80, 256 * KB)),
    ('halve the round trip', (size, 1000, 40, 256 * KB)),
    ('quadruple the window', (size, 1000, 80, 1 * MB)),
    ('two parallel streams',  (size, 1000, 80, 512 * KB)),
]:
    ceiling, seconds = report(*args)
    print(f'{label:<20}{" " * 34}-> {ceiling:>7,.1f} Mbit/s, {seconds:>6,.1f} s')

Doubling the bandwidth changes nothing at all, because the window was the constraint and it did not
move. This is the single most common misdiagnosis in a slow-transfer ticket: the path is upgraded,
the throughput is identical, and the upgrade is blamed on the provider.

Parallel streams appear in the table because they are the workaround an application reaches for
when it cannot change the window: two connections each with their own window add up.

## What to try next

- Work out the window your own production path needs: measure the round trip with `ping`, take the
  link rate from the instance type, and compare the product against the socket buffer sizes your
  host actually has configured.
- Add a loss term. At a loss rate `p`, a classic congestion-control algorithm settles near
  `MSS / (RTT * sqrt(p))`, which is a much harsher limit than the window on a long path. The
  congestion-window notebook in this directory builds that.
- Model a request that depends on another request. Two sequential round trips cost twice the
  latency and no extra bandwidth, which is why removing a round trip from a handshake is worth more
  than compressing the payload.